<a href="https://colab.research.google.com/github/leklek13/probable-octo-succotash/blob/my-first-branch/Extrator_de_Dados_Jur%C3%ADdicos_Extra%C3%A7%C3%A3o_Aprimorada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import tkinter as tk
from tkinter import filedialog, scrolledtext, messagebox
import PyPDF2

# --- CONFIGURAÇÃO IMPORTANTE ---
# Defina o caminho para o executável do Tesseract-OCR.
# Exemplo: r"C:\Program Files\Tesseract-OCR\tesseract.exe"
# Se o Tesseract não estiver no PATH do sistema, descomente e ajuste a linha abaixo:
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Defina o caminho para a pasta 'bin' do Poppler.
# Necessário para converter PDF em imagem (usado pelo pytesseract para PDFs de imagem).
# Exemplo: r"C:\poppler-23.11.0\Library\bin"
# Se o Poppler não estiver no PATH do sistema, descomente e ajuste a linha abaixo:
# POPPLER_PATH = r"C:\poppler-23.11.0\Library\bin"
POPPLER_PATH = None # Defina None se o Poppler estiver no PATH ou ajuste o caminho acima

def extrair_info_essencial(texto_original):
    """
    Extrai informações específicas de um texto usando expressões regulares aprimoradas.
    """
    # Normaliza múltiplos espaços e remove espaços no início/fim.
    # Preserva quebras de linha para algumas extrações, mas também cria versões sem para outras.
    texto_processado_com_quebras = re.sub(r' +', ' ', texto_original).strip()
    texto_processado_sem_quebras = re.sub(r'\s+', ' ', texto_original).strip()

    info = {}

    def buscar(padroes, texto_interno, grupo=1, flags=re.IGNORECASE | re.DOTALL):
        for padrao in padroes:
            match = re.search(padrao, texto_interno, flags)
            if match:
                try:
                    # Tenta pegar o grupo especificado
                    valor_capturado = match.group(grupo)
                    if valor_capturado is not None:
                        return valor_capturado.strip()
                    # Se o grupo especificado for None, mas o grupo 0 (match completo) existir,
                    # e o padrão não for apenas um placeholder como (.*?), pode ser útil.
                    # Mas geralmente, se o grupo de captura é None, queremos "Não identificado".
                    return "" # Retorna string vazia para ser tratada como "Não identificado" depois
                except IndexError:
                    # Se o grupo não existe (ex: regex sem parênteses de captura para esse grupo)
                    # e estamos pedindo grupo > 0, é um erro no padrão.
                    # Se pedimos grupo 0 (ou padrão), e ele existe:
                    if grupo == 0 and match.group(0) is not None:
                         return match.group(0).strip()
                    return "" # Retorna string vazia
        return "Não identificado" # Se nenhum padrão encontrar nada

    # --- Extração de campos ---

    # Data do Pedido
    info["Data do Pedido"] = buscar([
        r"(?:Pede\s+deferimento\.?|\S+,\s*|Local e data:)\s*([A-ZÀ-Ú][a-zà-ú]+\s*(?:d[aeo]\s*[A-ZÀ-Ú][a-zà-ú]+)?,\s*\d{1,2}\s+de\s+[a-zA-Zà-úÀ-Ú]+\s+de\s+\d{4})", # "Cidade, DD de Mes de AAAA"
        r"protocolado\s+em\s+(\d{1,2}\/\d{1,2}\/\d{4})",
        r"em\s+(\d{1,2}\s+de\s+[a-zA-Zà-úÀ-Ú]+\s+de\s+\d{4})", # "em DD de Mes de AAAA"
        r"(\d{1,2}\s+de\s+[a-zA-Zà-úÀ-Ú]+\s+de\s+\d{4})" # Formato solto
    ], texto_processado_sem_quebras)


    # 1. Número do Processo
    info["1. Número do Processo"] = buscar([
        r"Processo\s*(?:nº|Nº|n\.|N\.|:|n[úu]mero)?\s*([\d\.\-\/]{7,25})", # Mais comum
        r"Autos\s*(?:nº|Nº|n\.|N\.|:)?\s*([\d\.\-\/]{7,25})",
        r"protocolo\s*(?:nº|Nº|n\.|N\.|:|sob o nº)?\s*([\d\.\-\/]{7,25})",
        r"Ref\s*[:\.]?\s*Processo\s*([\d\.\-\/]{7,25})",
        r"\b([\d]{7}\-[\d]{2}\.[\d]{4}\.[\d]\.[\d]{2}\.[\d]{4})\b" # Formato CNJ específico
    ], texto_processado_sem_quebras)


    # 3. Autor(a) - Nome Completo
    # Tenta capturar nomes em maiúsculas, antes de qualificações comuns.
    # Usa texto_processado_com_quebras para tentar pegar nomes que podem estar em linhas diferentes da qualificação.
    autor_nome_raw = buscar([
        # Padrão mais robusto tentando pegar o nome antes da qualificação detalhada
        r"([A-ZÀ-Ú][A-ZÀ-Ú\s\.\-,'()]+?),\s*(?:brasileir[ao]|nacionalidade|CPF\s*nº|RG\s*nº|casad[ao]|solteir[ao]|vi[úu]v[ao]|menor|incapaz|aposentad[ao]|[\w\s]+portador)",
        # Padrão para "AUTOR(A)/REQUERENTE: NOME COMPLETO"
        r"^(?:AUTOR(?:A)?|REQUERENTE|EXEQUENTE)\s*:\s*([A-ZÀ-Ú][A-ZÀ-Ú\s\.\-]+)",
        # Padrão genérico se o nome estiver no início do documento, seguido por vírgula e qualificação
        r"^([A-ZÀ-Ú][A-ZÀ-Ú\s\.\-]+?),\s*(?:já\s+qualificad[ao]|devidamente\s+qualificad[ao])",
        # Padrão para "NOME, qualificação..., vem..."
        r"([A-ZÀ-Ú][A-ZÀ-Ú\s\.\-]+?),\s*(?:[\w\s]+,\s*){1,5}(?:vem\s+respeitosamente|prop[oô]r|ajuizar)"
    ], texto_processado_com_quebras, flags=re.IGNORECASE | re.MULTILINE) # Multiline para ^

    if autor_nome_raw != "Não identificado" and autor_nome_raw:
        # Limpeza adicional: remover termos comuns que podem ser capturados erroneamente
        autor_nome_limpo = re.sub(r'(?:PRIORIDADE DE TRAMITAÇÃO|PESSOA IDOSA|JUSTIÇA GRATUITA|EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO|POLO ATIVO)\s*[:\-]?\s*', '', autor_nome_raw, flags=re.IGNORECASE)
        autor_nome_limpo = re.sub(r'\s*\(.*?\)\s*', '', autor_nome_limpo) # Remove conteúdo entre parênteses (ex: (MENOR))
        autor_nome_limpo = re.sub(r'\s*,\s*$', '', autor_nome_limpo.strip()).strip() # Remove vírgula no final
        info["3. Autor(a) - Nome Completo"] = autor_nome_limpo if autor_nome_limpo else "Não identificado"
    else:
        info["3. Autor(a) - Nome Completo"] = "Não identificado"


    # 4. Autor(a) - CPF
    # Busca por CPFs formatados ou não, associados a palavras-chave.
    cpfs_encontrados = re.findall(
        r"(?:CPF|nº|inscrit[ao]\s+no\s+CPF\s*(?:sob\s+o\s+n[º\.]?)?)\s*[:\s]*(\d{3}\.?\d{3}\.?\d{3}-?\d{2}|\d{11})",
        texto_processado_com_quebras, re.IGNORECASE
    )
    cpfs_limpos = set()
    for cpf in cpfs_encontrados:
        cpf_numeros = re.sub(r'[^\d]', '', cpf)
        if len(cpf_numeros) == 11:
            cpfs_limpos.add(f"{cpf_numeros[:3]}.{cpf_numeros[3:6]}.{cpf_numeros[6:9]}-{cpf_numeros[9:]}")
    info["4. Autor(a) - CPF"] = "; ".join(sorted(list(cpfs_limpos))) if cpfs_limpos else "Não identificado"


    # 5. Autor(a) - RG
    rgs_encontrados = re.findall(
        r"(?:RG|identidade|C\.I\.)\s*(?:nº|Nº|n\.|N\.|:|sob\s+o\s+n[º\.]?)?\s*([\d\.X\s\-\/A-Z]{5,15}(?:SSP|PC|IFP|DETRAN|SEJUSP)?[A-Z\/\-]{0,4}[A-Z]{0,2})",
        texto_processado_com_quebras, re.IGNORECASE
    )
    rgs_limpos = set()
    for rg in rgs_encontrados:
        rg_tratado = rg.strip()
        # Evitar capturar partes de números de processo ou outros códigos como RG
        if not (len(re.findall(r'\.', rg_tratado)) > 3 or len(re.findall(r'\/', rg_tratado)) > 2):
             rgs_limpos.add(rg_tratado)
    info["5. Autor(a) - RG"] = "; ".join(sorted(list(rgs_limpos))) if rgs_limpos else "Não identificado"


    # Telefone ou Celular do Autor
    # Procura por telefones próximos ao nome do autor ou palavras como "contato", "telefone".
    # Evita números de OAB e processo.
    info["Telefone/Celular do Autor"] = "Não identificado"
    nome_autor_para_busca = info["3. Autor(a) - Nome Completo"].split(";")[0].strip() if info["3. Autor(a) - Nome Completo"] != "Não identificado" else "AUTORXXYYZZ"

    # Tenta buscar perto do nome do autor primeiro
    match_telefone_autor = re.search(
        rf"{re.escape(nome_autor_para_busca)}.*?(?:telefone|celular|contato|tel\.?)\s*(?:n[º\.]?)?\s*[:\s\-]*((?:\(?\d{2}\)?\s*)?(?:9\d{4}|\d{4})[\-\s\.]?\d{4})\b",
        texto_processado_com_quebras, re.IGNORECASE | re.DOTALL
    )
    if match_telefone_autor:
        telefone = match_telefone_autor.group(1).strip()
        if not re.search(r"OAB", telefone, re.IGNORECASE) and len(re.sub(r'\D', '', telefone)) >= 8:
            info["Telefone/Celular do Autor"] = telefone
    else:
        # Busca genérica por telefones, depois verifica se não é de advogado
        telefones_gerais = re.findall(
            r"\b((?:\(?\d{2}\)?\s*)?(?:9\d{4}|\d{4})[\-\s\.]?\d{4})\b",
            texto_processado_com_quebras
        )
        adv_texto_bloco = "".join(re.findall(r"Advogad[ao][\s\S]{0,100}", texto_processado_com_quebras, re.IGNORECASE))

        for tel in telefones_gerais:
            if not re.search(r"OAB", tel, re.IGNORECASE) and len(re.sub(r'\D', '', tel)) >= 8:
                 # Se o telefone não estiver no bloco de texto do advogado, assume que é do autor
                 if tel not in adv_texto_bloco:
                    info["Telefone/Celular do Autor"] = tel
                    break


    # 2. Tipo de Ação
    # Usa texto_processado_com_quebras para capturar nomes de ação que podem ter quebras de linha.
    tipo_acao_raw = buscar([
        # Padrão principal: "propor a presente AÇÃO DE ..."
        r"propor\s+(?:a\s+presente)?\s*(A[ÇC][ÃA]O\s+D[EO]\s+[\w\sÀ-Úà-ú\-\/\(\)]+)(?:\s+c(?:umulada|\/)?c|\s+com\s+pedido|\s+em\s+face|\s+contra|\s+pelos\s+fatos|\s+em\s+desfavor)",
        # Padrão para "ajuizar AÇÃO DE ..."
        r"ajuizar\s*(A[ÇC][ÃA]O\s+D[EO]\s+[\w\sÀ-Úà-ú\-\/\(\)]+)(?:\s+c(?:umulada|\/)?c|\s+com\s+pedido|\s+em\s+face|\s+contra|\s+pelos\s+fatos|\s+em\s+desfavor)",
        # Padrão para "Trata-se de AÇÃO DE ..."
        r"Trata-se\s+de\s*(A[ÇC][ÃA]O\s+D[EO]\s+[\w\sÀ-Úà-ú\-\/\(\)]+)",
        # Padrão para AÇÃO DE ... (no início de uma linha, em maiúsculas)
        r"^\s*(A[ÇC][ÃA]O\s+D[EO]\s+[\w\sÀ-Úà-ú\-\/\(\)]+)\s*$",
        # Padrão mais genérico se os outros falharem, buscando por "AÇÃO DE" seguido por palavras em maiúsculo.
        r"(A[ÇC][ÃA]O\s+D[EO]\s+(?:[A-ZÀ-Ú][A-ZÀ-Ú\s\/\(\)]+)+)"
    ], texto_processado_com_quebras, flags=re.IGNORECASE | re.MULTILINE)

    if tipo_acao_raw != "Não identificado" and tipo_acao_raw:
        # Limpa termos como "COM PEDIDO DE TUTELA DE URGÊNCIA" se fizerem parte da captura principal
        tipo_acao_limpo = re.sub(r"\s+COM\s+PEDIDO\s+DE.*$", "", tipo_acao_raw, flags=re.IGNORECASE).strip()
        tipo_acao_limpo = re.sub(r"\s+C\/C.*$", "", tipo_acao_limpo, flags=re.IGNORECASE).strip()
        info["2. Tipo de Ação"] = tipo_acao_limpo.upper() # Padroniza para maiúsculas
    else:
        info["2. Tipo de Ação"] = "Não identificado"


    # 6. Autor(a) - Qualificação/Condição Relevante
    qualificacoes = set()
    if re.search(r"pessoa\s+idosa|maior\s+de\s+60\s+anos|idade\s+igual\s+ou\s+superior\s+a\s+60", texto_processado_sem_quebras, re.IGNORECASE):
        qualificacoes.add("Pessoa Idosa (Prioridade)")
    if re.search(r"portador(?:a)?\s+de\s+doen[çc]a\s+grave|neoplasia\s+maligna|c[âa]ncer", texto_processado_sem_quebras, re.IGNORECASE):
        qualificacoes.add("Portador(a) de Doença Grave")
    if re.search(r"benefici[áa]ri[ao]\s+d[oa]\s+(?:BPC|LOAS|Benef[íi]cio\s+de\s+Presta[çc][ãa]o\s+Continuada)", texto_processado_sem_quebras, re.IGNORECASE):
        qualificacoes.add("Beneficiário(a) do BPC/LOAS")
    if re.search(r"vi[úu]v[ao]", texto_processado_sem_quebras, re.IGNORECASE): qualificacoes.add("Viúvo(a)")
    if re.search(r"aposentad[ao]", texto_processado_sem_quebras, re.IGNORECASE): qualificacoes.add("Aposentado(a)")
    if re.search(r"pensionista", texto_processado_sem_quebras, re.IGNORECASE): qualificacoes.add("Pensionista")
    if re.search(r"servidor(?:a)?\s+p[úu]blic[ao]", texto_processado_sem_quebras, re.IGNORECASE): qualificacoes.add("Servidor(a) Público(a)")
    if re.search(r"do\s+lar", texto_processado_sem_quebras, re.IGNORECASE): qualificacoes.add("Do Lar")
    if re.search(r"estudante", texto_processado_sem_quebras, re.IGNORECASE): qualificacoes.add("Estudante")
    if re.search(r"desempregad[ao]", texto_processado_sem_quebras, re.IGNORECASE): qualificacoes.add("Desempregado(a)")
    if re.search(r"defici[êe]ncia|PCD|pessoa\s+com\s+defici[êe]ncia", texto_processado_sem_quebras, re.IGNORECASE): qualificacoes.add("Pessoa com Deficiência")

    representado_match = re.search(r"representad[ao]\s+(?:legalmente\s+)?por\s+s(?:eu|ua)\s+(genitora?|curador(?:a)?|tutor(?:a)?)\s*[,:]?\s*([A-ZÀ-Ú][A-ZÀ-Ú\s\.\-]+)", texto_processado_sem_quebras, re.IGNORECASE)
    if representado_match:
        qualificacoes.add(f"Representado por {representado_match.group(1)}: {representado_match.group(2).strip()}")

    info["6. Autor(a) - Qualificação/Condição Relevante"] = ", ".join(sorted(list(qualificacoes))) if qualificacoes else "Não identificado"


    # 12. Principais Pedidos (Definitivos)
    # Procura seções como "DOS PEDIDOS", "REQUERIMENTOS FINAIS" e extrai itens.
    # Esta é uma extração complexa e pode precisar de mais refinamento dependendo dos documentos.
    pedidos_bloco_match = re.search(r"(?:DOS\s+PEDIDOS|REQUERIMENTOS(?:\s+FINAIS)?|DO\s+PEDIDO|PEDIDOS\s+E\s+REQUERIMENTOS)([\s\S]+?)(?:Nestes\s+termos|Dá-se\s+à\s+causa|VALOR\s+DA\s+CAUSA|Local,\s+data|assinatura|ADVOGADO|OAB)", texto_processado_com_quebras, re.IGNORECASE)
    pedidos_definitivos = set()
    if pedidos_bloco_match:
        bloco = pedidos_bloco_match.group(1)
        # Tenta extrair itens numerados ou com marcadores
        itens_pedido = re.findall(r"^\s*[a-z\d][\.\)]\s+(.+?)(?=\n\s*[a-z\d][\.\)]|\n\n|$)", bloco, re.MULTILINE | re.IGNORECASE)
        if not itens_pedido: # Se não achar itens numerados, tenta pegar frases que parecem pedidos
            itens_pedido = re.findall(r"(?:seja\s+condenad[ao]|seja\s+declarad[ao]|requer\s+a\s+condena[çc][ãa]o|determine\s+que|intima[çc][ãa]o\s+d[oa]\s+r[ée]u\s+para[\s\S]+?)(?:[^\n]+)", bloco, re.IGNORECASE)

        for item in itens_pedido:
            pedido_limpo = re.sub(r'\s+', ' ', item).strip()
            if len(pedido_limpo) > 20: # Evita itens muito curtos/ruído
                 # Limita o tamanho do pedido para não poluir o resumo
                pedidos_definitivos.add(pedido_limpo[:200] + "..." if len(pedido_limpo) > 200 else pedido_limpo)

    # Adiciona alguns pedidos comuns baseados no tipo de ação, se não encontrados no bloco
    if not pedidos_definitivos and info["2. Tipo de Ação"] != "Não identificado":
        if "INDENIZAÇÃO" in info["2. Tipo de Ação"] or "DANOS MORAIS" in info["2. Tipo de Ação"]:
            pedidos_definitivos.add("Condenação por Danos Morais.")
        if "OBRIGAÇÃO DE FAZER" in info["2. Tipo de Ação"]:
            pedidos_definitivos.add("Cumprimento de obrigação de fazer.")
        if "DECLARATÓRIA" in info["2. Tipo de Ação"]:
            pedidos_definitivos.add("Declaração judicial (ex: inexistência de débito).")

    info["12. Principais Pedidos (Definitivos)"] = "; ".join(sorted(list(pedidos_definitivos))) if pedidos_definitivos else "Não identificado"


    # 11. Pedido de Tutela de Urgência (Liminar)
    info["11. Pedido de Tutela de Urgência (Liminar)"] = "Não"
    liminar_keywords_gerais = r"(?:TUTELA\s+(?:DE\s+URG[ÊE]NCIA|PROVIS[ÓO]RIA|ANTECIPADA)|LIMINARMENTE|PEDIDO\s+LIMINAR|ANTECIPA[ÇC][ÃA]O\s+DOS\s+EFEITOS\s+DA\s+TUTELA)"

    liminar_match = re.search(rf"{liminar_keywords_gerais}([\s\S]{{0,500}}?)(?:DOS\s+FATOS|DO\s+DIREITO|DOS\s+PEDIDOS|REQUERIMENTOS\s+FINAIS|M[ÉR]RITO)", texto_processado_com_quebras, re.IGNORECASE)
    if liminar_match:
        detalhe_liminar_raw = liminar_match.group(1).strip()
        # Tenta extrair o objetivo da liminar
        objetivo_liminar_match = re.search(r"(?:para\s+(?:que|determinar|impedir|suspender)|a\s+fim\s+de|consistente\s+em|objetivando|requer-se\s+a\s+concessão\s+para)\s+([^.,;\n]+(?:[\s,]+(?:e|ou)[^.,;\n]+)*)", detalhe_liminar_raw, re.IGNORECASE)
        if objetivo_liminar_match:
            detalhe = re.sub(r'\s+', ' ', objetivo_liminar_match.group(1)).strip()
            info["11. Pedido de Tutela de Urgência (Liminar)"] = f"Sim - {detalhe[:150]}" + ("..." if len(detalhe) > 150 else "")
        else:
            info["11. Pedido de Tutela de Urgência (Liminar)"] = "Sim - Detalhes a verificar no texto"
    elif re.search(liminar_keywords_gerais, texto_processado_sem_quebras, re.IGNORECASE): # Busca geral se a detalhada falhar
        info["11. Pedido de Tutela de Urgência (Liminar)"] = "Sim - Detalhes a verificar no texto"


    # 7. Ré(u) - Nome/Razão Social
    # Procura por "em face de", "contra", "requerido(a):"
    # Usa texto_processado_com_quebras para nomes que podem abranger múltiplas linhas.
    reu_bloco_match = re.search(r"(?:em\s+face\s+d[eao]|contra|em\s+desfavor\s+d[eao]|parte\s+r[ée]\s*:|REQUERID[OA]\s*:\s*)([\s\S]+?)(?:pelos\s+fatos|com\s+endere[çc]o|CNPJ|CPF|qualifica[çc][ãa]o|advogado|nesta\s+cidade|CEP)", texto_processado_com_quebras, re.IGNORECASE)
    reus_encontrados = set()
    if reu_bloco_match:
        bloco_reus = reu_bloco_match.group(1)
        # Tenta dividir por "e", ";", ou quebras de linha seguidas de nome em maiúscula (indicando múltiplos réus)
        potenciais_reus = re.split(r"\s+e\s+(?=[A-ZÀ-Ú])|\s*;\s*|\n\s*(?=[A-ZÀ-Ú])", bloco_reus)
        for reu_raw in potenciais_reus:
            reu_nome = re.match(r"^\s*([A-ZÀ-Ú][A-ZÀ-Ú\s\.\-,'()&]+(?:LTDA|S\.?A\.?|EIRELI|MEI|ME|CIA|BANK|BANCO|SEGURADORA|MUNIC[ÍI]PIO\s+DE\s+[\w\s]+|ESTADO\s+DE\s+[\w\s]+)?)\b", reu_raw.strip(), re.IGNORECASE)
            if reu_nome:
                nome_limpo = reu_nome.group(1).strip()
                # Remove qualificações comuns que podem ter sido pegas
                nome_limpo = re.sub(r",\s*(?:pessoa\s+jur[íi]dica.*|,?\s*inscrit[ao]\s+no\s+CNPJ.*|,?\s*com\s+sede.*)", "", nome_limpo, flags=re.IGNORECASE).strip()
                nome_limpo = re.sub(r"\s*\(.*?\)\s*", "", nome_limpo).strip() # Remove conteúdo entre parênteses
                if len(nome_limpo) > 3 : # Evita ruídos muito curtos
                    reus_encontrados.add(nome_limpo)

    if not reus_encontrados: # Fallback se o bloco não funcionar
        reu_simples = buscar([
            r"^(?:R[ÉE]U|REQUERID[AO]|EXECUTAD[AO])\s*:\s*([A-ZÀ-Ú][A-ZÀ-Ú\s\.\-]+)",
        ], texto_processado_com_quebras, flags=re.IGNORECASE | re.MULTILINE)
        if reu_simples != "Não identificado" and reu_simples:
            reus_encontrados.add(reu_simples)

    info["7. Ré(u) - Nome/Razão Social"] = "; ".join(sorted(list(reus_encontrados))) if reus_encontrados else "Não identificado"


    # 8. Ré(u) - CNPJ
    # Procura CNPJs próximos aos nomes dos réus ou de forma geral
    cnpjs_reu_encontrados = set()
    if reu_bloco_match: # Se achou bloco de réus, procura CNPJ dentro ou logo após
        bloco_cnpj_busca = reu_bloco_match.group(0) # Pega o bloco inteiro do match do réu
        cnpjs_raw = re.findall(r"(?:CNPJ|CGC)\s*(?:n[º\.]?|sob\s+o\s+n[º\.]?)?\s*[:\s\-]*((?:\d{2}\.?\d{3}\.?\d{3}\/?\d{4}-?\d{2})|\d{14})", bloco_cnpj_busca, re.IGNORECASE)
        for cnpj in cnpjs_raw:
            cnpj_numeros = re.sub(r'[^\d]', '', cnpj)
            if len(cnpj_numeros) == 14:
                cnpjs_reu_encontrados.add(f"{cnpj_numeros[:2]}.{cnpj_numeros[2:5]}.{cnpj_numeros[5:8]}/{cnpj_numeros[8:12]}-{cnpj_numeros[12:]}")

    if not cnpjs_reu_encontrados: # Busca geral se não encontrou no bloco do réu
         cnpjs_raw = re.findall(r"\b((?:\d{2}\.?\d{3}\.?\d{3}\/?\d{4}-?\d{2})|\d{14})\b", texto_processado_sem_quebras)
         for cnpj in cnpjs_raw:
            # Verifica se é realmente um CNPJ e não parte de outro número
            if texto_processado_sem_quebras.lower().count(cnpj.lower()) == 1 or "cnpj" in texto_processado_sem_quebras[max(0, texto_processado_sem_quebras.lower().find(cnpj.lower())-10) : texto_processado_sem_quebras.lower().find(cnpj.lower())].lower():
                cnpj_numeros = re.sub(r'[^\d]', '', cnpj)
                if len(cnpj_numeros) == 14:
                    cnpjs_reu_encontrados.add(f"{cnpj_numeros[:2]}.{cnpj_numeros[2:5]}.{cnpj_numeros[5:8]}/{cnpj_numeros[8:12]}-{cnpj_numeros[12:]}")

    info["8. Ré(u) - CNPJ"] = "; ".join(sorted(list(cnpjs_reu_encontrados))) if cnpjs_reu_encontrados else "Não identificado"


    # 10. Comarca/Vara
    # Busca por "VARA ... DA COMARCA DE ..." ou "COMARCA DE ... - VARA ..."
    comarca_vara_raw = buscar([
        r"((?:\d+[ªº]?|Primeira|Segunda|Terceira|Quarta|Quinta|Sexta|S[ée]tima|Oitava|Nona|D[ée]cima)\s+Vara\s+(?:C[íi]vel|Criminal|da\s+Fazenda\s+P[úu]blica|do\s+Trabalho|de\s+Fam[íi]lia|do\s+Juizado\s+Especial\s+C[íi]vel)?\s+(?:da\s+Comarca\s+d[eo]\s+|d[eo]\s+Foro\s+d[eo]\s+|Regional\s+d[eo]\s+)?\s*([A-ZÀ-Ú][A-ZÀ-Ú\s\-\/\(\)]+))",
        r"(Juizado\s+Especial\s+(?:C[íi]vel|Criminal|da\s+Fazenda\s+P[úu]blica)\s+(?:da\s+Comarca\s+d[eo]\s+|d[eo]\s+Foro\s+d[eo]\s+)?\s*([A-ZÀ-Ú][A-ZÀ-Ú\s\-\/\(\)]+))",
        r"Vara\s+[ÚU]nica\s+(?:da\s+Comarca\s+d[eo]\s+|d[eo]\s+Foro\s+d[eo]\s+)?\s*([A-ZÀ-Ú][A-ZÀ-Ú\s\-\/\(\)]+)",
        r"Comarca\s+d[eo]\s+([A-ZÀ-Ú][A-ZÀ-Ú\s\-\/\(\)]+)\s*-\s*((?:\d+[ªº]?)\s+Vara\s+(?:C[íi]vel|Criminal)?)",
        r"Tribunal\s+de\s+Justi[çc]a\s+d[eo]\s+Estado\s+d[eo]\s+([A-ZÀ-Ú][A-ZÀ-Ú\s\-\/\(\)]+)"
    ], texto_processado_sem_quebras, grupo=0) # Pega o match completo para limpar depois

    if comarca_vara_raw != "Não identificado" and comarca_vara_raw:
        # Limpeza de termos que podem vir antes ou depois
        comarca_vara_limpa = re.sub(r"Excelent[íi]ssimo\(a\)\s+Senhor\(a\)\s+Doutor\(a\)\s+Juiz\(a\)\s+de\s+Direito\s+d[a-o]\s+", "", comarca_vara_raw, flags=re.IGNORECASE).strip()
        comarca_vara_limpa = re.sub(r"\s*-\s*(?:MS|SP|RJ|MG|PR|SC|RS|BA|PE|CE|AM|PA|MT|GO|ES|AL|PI|RN|PB|SE|RO|TO|AC|AP|RR|DF)$", "", comarca_vara_limpa, flags=re.IGNORECASE).strip() # Remove sigla do estado no final
        info["10. Comarca/Vara"] = comarca_vara_limpa
    else:
        info["10. Comarca/Vara"] = "Não identificado"


    # 9. Valor da Causa
    info["9. Valor da Causa"] = buscar([
        r"(?:VALOR\s+D[AÀ]\s+CAUSA|D[áa]-se\s+[àa]\s+causa\s+o\s+valor\s+de|Atribui-se\s+[àa]\s+causa\s+o\s+valor\s+de)\s*[:\s]*R\$\s*([\d\.,]+)",
        r"valor\s*:\s*R\$\s*([\d\.,]+)" # Simples "valor: R$ ..."
    ], texto_processado_sem_quebras)


    # 13. Valor da Indenização por Danos Morais Solicitado
    info["13. Valor da Indenização por Danos Morais Solicitado"] = "Não se aplica"
    # Procura por "danos morais" e um valor R$ próximo
    dm_match = re.search(r"indeniza[çc][ãa]o\s+(?:por|pelos|a\s+t[íi]tulo\s+de)\s+danos\s+morais(?:.*?)(?:no\s+valor\s+de|em|quantia\s+de|fixad[ao]\s+em)\s+R\$\s*([\d\.,]+)", texto_processado_sem_quebras, re.IGNORECASE | re.DOTALL)
    if dm_match:
        valor = dm_match.group(1).replace('.', '').replace(',', '.')
        try:
            info["13. Valor da Indenização por Danos Morais Solicitado"] = f"R$ {float(valor):.2f}".replace('.', ',')
        except ValueError:
            info["13. Valor da Indenização por Danos Morais Solicitado"] = f"R$ {dm_match.group(1).strip()}" # Mantém como string se não puder converter
    else: # Fallback se o padrão acima não encontrar
        dm_simples_match = re.search(r"danos\s+morais\s+em\s+R\$\s*([\d\.,]+)", texto_processado_sem_quebras, re.IGNORECASE)
        if dm_simples_match:
            valor = dm_simples_match.group(1).replace('.', '').replace(',', '.')
            try:
                info["13. Valor da Indenização por Danos Morais Solicitado"] = f"R$ {float(valor):.2f}".replace('.', ',')
            except ValueError:
                 info["13. Valor da Indenização por Danos Morais Solicitado"] = f"R$ {dm_simples_match.group(1).strip()}"


    # 14. Justiça Gratuita Solicitada
    info["14. Justiça Gratuita Solicitada"] = "Não"
    if re.search(r"(?:JUSTI[ÇC]A\s+GRATUITA|GRATUIDADE\s+D[AE]\s+JUSTI[ÇC]A|benef[íi]cios\s+da\s+assist[êe]ncia\s+judici[áa]ria\s+gratuita|AJG|pleiteia\s+os\s+benef[íi]cios\s+da\s+justi[çc]a\s+gratuita)", texto_processado_sem_quebras, re.IGNORECASE):
        info["14. Justiça Gratuita Solicitada"] = "Sim"


    # 15. Prioridade de Tramitação Solicitada
    info["15. Prioridade de Tramitação Solicitada"] = "Não"
    prioridade_motivos = set()
    if re.search(r"(?:PRIORIDADE\s+DE\s+TRAMITA[ÇC][ÃA]O|TRAMITA[ÇC][ÃA]O\s+PRIORIT[ÁA]RIA|prioridade\s+no\s+tr[âa]mite)", texto_processado_sem_quebras, re.IGNORECASE):
        if "Pessoa Idosa (Prioridade)" in info["6. Autor(a) - Qualificação/Condição Relevante"]:
            prioridade_motivos.add("Pessoa Idosa")
        if "Portador(a) de Doença Grave" in info["6. Autor(a) - Qualificação/Condição Relevante"]:
            prioridade_motivos.add("Doença Grave")

        if prioridade_motivos:
            info["15. Prioridade de Tramitação Solicitada"] = f"Sim ({'; '.join(prioridade_motivos)})"
        else:
            info["15. Prioridade de Tramitação Solicitada"] = "Sim (Motivo a verificar no texto)"


    # 16. Advogados do Autor
    # Procura por nomes antes de "OAB" ou após "Advogado(a):"
    advogados_encontrados = set()
    # Padrão para nome seguido de OAB
    matches_adv_oab = re.findall(r"([A-ZÀ-Ú][A-ZÀ-Ú\s\.\-\']{5,50}?)\s*(?:ADVOGAD[AO]\s*\(A\)|ADV[A-Z\.]*)\s*OAB\s*[\/\-\s]*\s*([A-Z]{2})\s*n?[º\s\.]*([\d\.\/]+)", texto_processado_com_quebras, re.IGNORECASE)
    for match in matches_adv_oab:
        nome_adv = re.sub(r"(?:assinatura eletr[ôo]nica|documento assinado digitalmente).*", "", match[0], flags=re.IGNORECASE).strip()
        nome_adv = re.sub(r"\n", " ", nome_adv).strip() # Remove quebras de linha no nome
        if len(nome_adv.split()) > 1 and len(nome_adv) > 5 : # Nome razoável
            advogados_encontrados.add(f"{nome_adv} (OAB/{match[1].upper()} nº {match[2]})")

    # Padrão para "Advogado(a): NOME"
    if not advogados_encontrados:
        matches_adv_label = re.findall(r"Advogad[ao](?:s)?\s*\(as\)?\s*:\s*([A-ZÀ-Ú][A-ZÀ-Ú\s\.\-]+)", texto_processado_com_quebras, re.IGNORECASE)
        for nome_adv in matches_adv_label:
            advogados_encontrados.add(nome_adv.strip())

    info["16. Advogados do Autor"] = ", ".join(sorted(list(advogados_encontrados))) if advogados_encontrados else "Não identificado"


    # 17. E-mail para contato (advogado)
    # Procura e-mails próximos aos nomes dos advogados ou palavras-chave
    emails_adv = set()
    if advogados_encontrados: # Se encontrou advogados, procura e-mail perto deles
        for adv_str in advogados_encontrados:
            nome_adv_curto = adv_str.split('(')[0].strip().split()[-1] # Pega último sobrenome
            email_match = re.search(rf"{re.escape(nome_adv_curto)}[\s\S]{{0,100}}([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})", texto_processado_com_quebras, re.IGNORECASE)
            if email_match:
                emails_adv.add(email_match.group(1))

    # Busca geral por e-mails se não achou perto dos advogados
    if not emails_adv:
        todos_emails = re.findall(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b", texto_processado_com_quebras)
        palavras_chave_adv_email = ["adv", "juridico", "escritorio", "oab"]
        for email in todos_emails:
            # Verifica se o e-mail parece ser de advogado (contém palavras-chave ou está perto de "advogado")
            pos_email = texto_processado_com_quebras.lower().find(email.lower())
            texto_adjacente = texto_processado_com_quebras[max(0, pos_email-50) : pos_email+len(email)+50].lower()
            if any(keyword in email.lower() for keyword in palavras_chave_adv_email) or "advogad" in texto_adjacente:
                # Evita e-mails de réus (heurística simples)
                if not info["7. Ré(u) - Nome/Razão Social"] or not any(nome_reu.split()[0].lower() in email.lower() for nome_reu in info["7. Ré(u) - Nome/Razão Social"].split(';')):
                    emails_adv.add(email)

    info["17. E-mail para contato (advogado)"] = "; ".join(sorted(list(emails_adv))) if emails_adv else "Não identificado"

    # --- Construção do resumo formatado ---
    resumo_formatado = "==== RESUMO DO PROCESSO ====\n\n"
    ordem_campos = [
        "Data do Pedido", "1. Número do Processo",
        "3. Autor(a) - Nome Completo", "4. Autor(a) - CPF", "5. Autor(a) - RG", "Telefone/Celular do Autor",
        "2. Tipo de Ação", "6. Autor(a) - Qualificação/Condição Relevante",
        "12. Principais Pedidos (Definitivos)", "11. Pedido de Tutela de Urgência (Liminar)",
        "7. Ré(u) - Nome/Razão Social", "8. Ré(u) - CNPJ",
        "10. Comarca/Vara", "9. Valor da Causa",
        "13. Valor da Indenização por Danos Morais Solicitado",
        "14. Justiça Gratuita Solicitada", "15. Prioridade de Tramitação Solicitada",
        "16. Advogados do Autor", "17. E-mail para contato (advogado)"
    ]

    # Campos que devem ter quebra de linha dupla para melhor legibilidade
    campos_quebra_dupla = [
        "2. Tipo de Ação", "6. Autor(a) - Qualificação/Condição Relevante",
        "12. Principais Pedidos (Definitivos)", "11. Pedido de Tutela de Urgência (Liminar)"
    ]

    for chave in ordem_campos:
        valor = info.get(chave, "Não identificado")
        if not valor or valor.isspace(): # Se valor for string vazia ou só espaços
            valor = "Não identificado"

        if chave in campos_quebra_dupla:
            resumo_formatado += f"{chave}:\n{valor}\n\n"
        else:
            resumo_formatado += f"{chave}: {valor}\n"

    return resumo_formatado.strip()


def processar_arquivo(caminho_arquivo):
    ext = os.path.splitext(caminho_arquivo)[-1].lower()
    texto_extraido = ""
    print(f"Processando arquivo: {caminho_arquivo}")

    if ext == ".pdf":
        try:
            print("Tentando extração direta de texto do PDF...")
            with open(caminho_arquivo, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                if reader.is_encrypted:
                    try:
                        reader.decrypt('') # Tenta descriptografar com senha vazia
                        print("PDF descriptografado com senha vazia.")
                    except Exception as decrypt_error:
                        print(f"PDF criptografado. Tentativa de descriptografia falhou: {decrypt_error}.")

                if not reader.pages:
                    raise ValueError("PDF contém 0 páginas ou está vazio.")

                for page_num in range(len(reader.pages)):
                    page = reader.pages[page_num]
                    texto_pagina = page.extract_text()
                    if texto_pagina:
                        texto_extraido += texto_pagina + "\n"

                if texto_extraido and not texto_extraido.isspace():
                    print("Extração direta de texto do PDF bem-sucedida.")
                else:
                    print("Extração direta não retornou texto ou PDF é baseado em imagem. Recorrendo ao OCR.")
                    texto_extraido = "" # Limpa para garantir que o OCR seja usado
                    # Tenta OCR
                    if POPPLER_PATH and not os.path.exists(os.path.join(POPPLER_PATH, "pdftoppm.exe")): # Verifica se o caminho do Poppler é válido
                         print(f"[AVISO] Caminho do Poppler configurado ({POPPLER_PATH}) mas pdftoppm.exe não encontrado lá. OCR de PDF pode falhar.")

                    imagens = convert_from_path(caminho_arquivo, poppler_path=POPPLER_PATH)
                    for i, imagem in enumerate(imagens):
                        print(f"Fazendo OCR da página {i+1}/{len(imagens)} do PDF...")
                        texto_extraido += pytesseract.image_to_string(imagem, lang='por') + "\n"
                    if texto_extraido and not texto_extraido.isspace():
                        print("OCR do PDF concluído.")
                    else:
                        print("OCR do PDF não retornou texto.")

        except PyPDF2.errors.PdfReadError as pe:
             print(f"Erro PyPDF2 ao ler PDF (pode estar corrompido ou ser apenas imagem): {pe}. Tentando OCR...")
             texto_extraido = ""
             try:
                 if POPPLER_PATH and not os.path.exists(os.path.join(POPPLER_PATH, "pdftoppm.exe")):
                     print(f"[AVISO] Caminho do Poppler configurado ({POPPLER_PATH}) mas pdftoppm.exe não encontrado lá. OCR de PDF pode falhar.")
                 imagens = convert_from_path(caminho_arquivo, poppler_path=POPPLER_PATH)
                 for i, imagem in enumerate(imagens):
                     print(f"Fazendo OCR da página {i+1}/{len(imagens)} do PDF (fallback)...")
                     texto_extraido += pytesseract.image_to_string(imagem, lang='por') + "\n"
                 if texto_extraido and not texto_extraido.isspace():
                     print("OCR do PDF (fallback) concluído.")
                 else:
                     print("OCR do PDF (fallback) não retornou texto.")
             except Exception as e_ocr:
                 msg_erro = f"[ERRO] Falha crítica ao processar PDF com OCR (PyPDF2 já havia falhado): {e_ocr}"
                 print(msg_erro)
                 return msg_erro, "" # Retorna erro e texto vazio
        except Exception as e:
            msg_erro = f"[ERRO] Falha ao processar PDF: {e}"
            print(msg_erro)
            # Tenta OCR como último recurso para erros genéricos de PDF
            if not isinstance(e, PyPDF2.errors.PdfReadError) and "PDF contém 0 páginas" not in str(e):
                try:
                    print("Tentando OCR como fallback para erro desconhecido no PDF...")
                    if POPPLER_PATH and not os.path.exists(os.path.join(POPPLER_PATH, "pdftoppm.exe")):
                        print(f"[AVISO] Caminho do Poppler configurado ({POPPLER_PATH}) mas pdftoppm.exe não encontrado lá. OCR de PDF pode falhar.")
                    imagens = convert_from_path(caminho_arquivo, poppler_path=POPPLER_PATH)
                    texto_ocr_fallback = ""
                    for i, imagem in enumerate(imagens):
                        print(f"Fazendo OCR da página {i+1}/{len(imagens)} do PDF (fallback extremo)...")
                        texto_ocr_fallback += pytesseract.image_to_string(imagem, lang='por') + "\n"
                    texto_extraido = texto_ocr_fallback
                    if texto_extraido and not texto_extraido.isspace():
                        print("OCR do PDF (fallback extremo) concluído.")
                    else:
                        print("OCR do PDF (fallback extremo) não retornou texto.")
                except Exception as e_ocr_final:
                    msg_erro_final = f"[ERRO] Falha crítica final ao processar PDF com OCR: {e_ocr_final}"
                    print(msg_erro_final)
                    return msg_erro_final, ""
            else:
                return msg_erro, ""
    elif ext in [".png", ".jpg", ".jpeg", ".tiff", ".bmp", ".gif"]:
        try:
            print(f"Fazendo OCR da imagem: {caminho_arquivo}")
            imagem = Image.open(caminho_arquivo)
            texto_extraido = pytesseract.image_to_string(imagem, lang='por')
            print("OCR da imagem concluído.")
        except Exception as e:
            msg_erro = f"[ERRO] Falha ao processar imagem: {e}"
            print(msg_erro)
            return msg_erro, ""
    elif ext == ".txt":
        try:
            print(f"Lendo arquivo de texto: {caminho_arquivo}")
            with open(caminho_arquivo, "r", encoding="utf-8") as f:
                texto_extraido = f.read()
            print("Leitura do arquivo de texto concluída.")
        except Exception as e:
            msg_erro = f"[ERRO] Falha ao ler TXT: {e}"
            print(msg_erro)
            return msg_erro, ""
    else:
        msg_erro = f"[ERRO] Formato de arquivo '{ext}' não suportado."
        print(msg_erro)
        return msg_erro, ""

    if not texto_extraido or texto_extraido.isspace():
        msg_aviso = "[AVISO] Nenhum texto foi extraído do arquivo. O arquivo pode estar vazio, ser ilegível ou o OCR falhou."
        print(msg_aviso)
        # Retorna o aviso e o texto (que estará vazio ou com espaços)
        return msg_aviso, texto_extraido

    print("Extraindo informações essenciais...")
    resumo_formatado = extrair_info_essencial(texto_extraido)
    print("Extração de informações concluída.")
    # Retorna o resumo e o texto completo extraído
    return resumo_formatado, texto_extraido

def selecionar_arquivo_e_processar():
    caminho = filedialog.askopenfilename(
        title="Selecione um arquivo PDF, de Imagem ou TXT",
        filetypes=[("Arquivos Suportados", "*.pdf *.png *.jpg *.jpeg *.tiff *.bmp *.gif *.txt"),
                   ("Todos os arquivos", "*.*")]
    )
    if caminho:
        btn_selecionar.config(text="Processando...", state=tk.DISABLED)
        janela.update_idletasks()

        resultado_processamento, texto_completo = processar_arquivo(caminho)

        # Verifica se o resultado_processamento é uma mensagem de erro ou aviso
        if resultado_processamento.startswith("[ERRO]") or resultado_processamento.startswith("[AVISO]"):
            conteudo_para_exibir = f"{resultado_processamento}\n\n==== TEXTO COMPLETO (SE HOUVER) ====\n{texto_completo if texto_completo else 'Nenhum texto disponível.'}"
            salvar_arquivo = False
            if resultado_processamento.startswith("[ERRO]"):
                 messagebox.showerror("Erro no Processamento", resultado_processamento)
            else: # Aviso
                 messagebox.showwarning("Aviso no Processamento", resultado_processamento)

        else: # Sucesso na extração do resumo
            conteudo_para_exibir = f"{resultado_processamento}\n\n==== TEXTO COMPLETO ====\n{texto_completo}"
            salvar_arquivo = True

        texto_resultado_widget.config(state=tk.NORMAL)
        texto_resultado_widget.delete(1.0, tk.END)
        texto_resultado_widget.insert(tk.END, conteudo_para_exibir)
        texto_resultado_widget.config(state=tk.DISABLED)

        if salvar_arquivo:
            nome_base, _ = os.path.splitext(caminho)
            caminho_saida_txt = nome_base + "_analisado.txt"
            try:
                with open(caminho_saida_txt, "w", encoding="utf-8") as f_out:
                    # Salva o resumo e o texto completo no arquivo
                    f_out.write(conteudo_para_exibir)
                print(f"Resultado também salvo em: {caminho_saida_txt}")
                messagebox.showinfo("Arquivo Salvo",
                                    f"O resumo e o texto completo foram salvos em:\n{caminho_saida_txt}")
            except Exception as e:
                print(f"[ERRO] Não foi possível salvar o arquivo .txt: {e}")
                messagebox.showerror("Erro ao Salvar",
                                     f"Não foi possível salvar o arquivo .txt:\n{caminho_saida_txt}\n\nErro: {e}")

        btn_selecionar.config(text="Selecionar Outro Arquivo", state=tk.NORMAL)
    else:
        print("Nenhum arquivo selecionado.")

# --- Interface Gráfica (GUI) ---
janela = tk.Tk()
janela.title("Extrator de Dados Jurídicos (Extração Aprimorada v1.0)")
janela.geometry("850x650")

# Frame para o botão
frame_botoes = tk.Frame(janela)
frame_botoes.pack(pady=15)

btn_selecionar = tk.Button(frame_botoes, text="Selecionar Arquivo", command=selecionar_arquivo_e_processar, font=("Arial", 13), padx=12, pady=6, bg="#E0E0E0")
btn_selecionar.pack()

# Frame para a área de texto com scroll
frame_texto = tk.Frame(janela)
frame_texto.pack(padx=15, pady=(0,15), fill=tk.BOTH, expand=True)

texto_resultado_widget = scrolledtext.ScrolledText(frame_texto, width=100, height=30, wrap=tk.WORD, state=tk.DISABLED, font=("Arial", 10))
texto_resultado_widget.pack(fill=tk.BOTH, expand=True)

# Adiciona uma verificação inicial do Tesseract e Poppler (opcional, mas útil para o usuário)
try:
    pytesseract.get_tesseract_version()
    print("Tesseract-OCR encontrado e funcionando.")
except Exception as e:
    print(f"[CONFIGURAÇÃO] Tesseract-OCR não encontrado ou não configurado corretamente: {e}")
    print("Verifique se o Tesseract está instalado e se pytesseract.pytesseract.tesseract_cmd está definido corretamente no script, se necessário.")

if POPPLER_PATH:
    if not os.path.isdir(POPPLER_PATH) or not os.path.exists(os.path.join(POPPLER_PATH, "pdftoppm.exe")):
        print(f"[CONFIGURAÇÃO] Caminho do Poppler ({POPPLER_PATH}) parece inválido ou 'pdftoppm.exe' não encontrado.")
        print("Verifique se o Poppler está instalado e se POPPLER_PATH está definido corretamente no script.")
    else:
        print(f"Poppler encontrado em: {POPPLER_PATH}")
else:
    print("[CONFIGURAÇÃO] POPPLER_PATH não está definido. O script tentará usar o Poppler do PATH do sistema. Se a conversão de PDF para imagem falhar, defina POPPLER_PATH.")


print("Interface gráfica iniciada. Aguardando seleção de arquivo.")
janela.mainloop()

ModuleNotFoundError: No module named 'pytesseract'